In [1]:
import asyncio
import hashlib
import json
import os
from collections import defaultdict
from collections.abc import Callable
from datetime import datetime
from typing import Any

from dotenv import load_dotenv
from google.genai import Client

load_dotenv()

True

In [2]:
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

google_client = Client(api_key=GOOGLE_API_KEY)
MODEL = "gemini-3.5-flash-lite"

### Task 1

In [3]:
def get_time() -> str:
    return datetime.now().astimezone().isoformat(timespec="seconds")


def calculator(a: float, b: float, operation: str) -> float:
    operations = {
        "add": lambda: a + b,
        "subtract": lambda: a - b,
        "multiply": lambda: a * b,
        "divide": lambda: (
            a / b if b != 0 else (_ for _ in ()).throw(ValueError("Cannot divide by zero."))
        ),
    }

    if operation not in operations:
        raise ValueError(f"Invalid operation: {operation}")

    return operations[operation]()


TOOL_DEFINITIONS = [
    {
        "type": "function",
        "name": "get_time",
        "description": "Get the current local date and time.",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
        },
    },
    {
        "type": "function",
        "name": "calculator",
        "description": "Perform basic arithmetic.",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {
                    "type": "number",
                    "description": "First number",
                },
                "b": {
                    "type": "number",
                    "description": "Second number",
                },
                "operation": {
                    "type": "string",
                    "enum": [
                        "add",
                        "subtract",
                        "multiply",
                        "divide",
                    ],
                },
            },
            "required": ["a", "b", "operation"],
        },
    },
]

TOOL_REGISTRY: dict[str, Callable[..., Any]] = {
    "get_time": get_time,
    "calculator": calculator,
}

In [4]:
async def run_agent(
    goal: str,
    client: Any,
    model: str,
    limit: dict[str, int] | None = None,
) -> str:
    limit = limit or {
        "max_steps": 5,
        "max_tool_calls": 10,
    }

    max_steps = limit["max_steps"]
    remaining_calls = limit["max_tool_calls"]

    if max_steps < 1 or remaining_calls < 0:
        raise ValueError("Invalid agent limits.")

    interactions = []

    interaction = client.interactions.create(
        model=model,
        input=goal,
        tools=TOOL_DEFINITIONS,
    )

    interactions.append(interaction)

    for step_number in range(max_steps):
        tool_calls = [step for step in (interaction.steps or []) if step.type == "function_call"]

        # No function calls means the model has finished.
        if not tool_calls:
            return interaction.output_text or ""

        if len(tool_calls) > remaining_calls:
            raise RuntimeError("Maximum tool-call limit exceeded.")

        remaining_calls -= len(tool_calls)
        tool_results = []

        # Execute every requested tool.
        for call in tool_calls:
            print(f"Tool requested: {call.name}")
            print(f"Arguments: {call.arguments}")

            if call.name not in TOOL_REGISTRY:
                raise ValueError(f"Unknown tool requested: {call.name}")

            try:
                result = TOOL_REGISTRY[call.name](**(call.arguments or {}))
                result_payload = {
                    "ok": True,
                    "result": result,
                }
            except Exception as exc:  # noqa
                result_payload = {
                    "ok": False,
                    "error": str(exc),
                }

            print(f"Tool result: {result_payload}")

            tool_results.append(
                {
                    "type": "function_result",
                    "name": call.name,
                    "call_id": call.id,
                    "result": [
                        {
                            "type": "text",
                            "text": json.dumps(result_payload),
                        }
                    ],
                }
            )

        # Return actual function results to the model.
        interaction = client.interactions.create(
            model=model,
            previous_interaction_id=interaction.id,
            input=tool_results,
            tools=TOOL_DEFINITIONS,
        )

        interactions.append(interaction)

    raise RuntimeError("Maximum agent steps exceeded before a final answer.")

In [5]:
result = await run_agent(
    goal=(
        "Calculate the sum of 5 and 10, then get the "
        "current time. Report both results in one answer."
    ),
    client=google_client,
    model=MODEL,
    limit={
        "max_steps": 5,
        "max_tool_calls": 10,
    },
)

print("\nFINAL ANSWER:")
print(result)

Tool requested: calculator
Arguments: {'b': 10, 'operation': 'add', 'a': 5}
Tool result: {'ok': True, 'result': 15}
Tool requested: get_time
Arguments: {}
Tool result: {'ok': True, 'result': '2026-10-15T12:27:35+05:30'}

FINAL ANSWER:
The sum of 5 and 10 is 15. The current time is October 15, 2026, at 12:27:35 PM (+05:30).


### Task 2

implemented in test_agent_limits.py

### Task 3

In [6]:
def hash_tool_call(name: str, arguments: dict[str, Any] | None) -> str:
    """Fingerprint (tool, canonical_args) by hashing name and sorted json arguments."""
    canonical_args = json.dumps(arguments or {}, sort_keys=True)
    return hashlib.sha256(f"{name}:{canonical_args}".encode()).hexdigest()


# Deliberately failing tool to trigger a retry cycle
def flaky_weather(city: str) -> str:
    raise RuntimeError(f"Service 503: Weather service failed for '{city}'. Please retry.")


TOOL_REGISTRY["flaky_weather"] = flaky_weather
TOOL_DEFINITIONS.append(
    {
        "type": "function",
        "name": "flaky_weather",
        "description": "Get weather for a city.",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string", "description": "City name"}},
            "required": ["city"],
        },
    }
)

In [7]:
async def run_agent_with_loop_detection(
    goal: str,
    client: Any,
    model: str,
    max_repeats: int = 2,
    max_steps: int = 6,
) -> dict[str, Any]:
    call_counts: dict[str, int] = defaultdict(int)

    interaction = client.interactions.create(
        model=model,
        input=goal,
        tools=TOOL_DEFINITIONS,
    )

    for step_number in range(max_steps):
        tool_calls = [step for step in (interaction.steps or []) if step.type == "function_call"]

        if not tool_calls:
            return {"status": "completed", "output": interaction.output_text}

        tool_results = []
        for call in tool_calls:
            # 1. Compute fingerprint hash for loop detection
            call_hash = hash_tool_call(call.name, call.arguments)
            call_counts[call_hash] += 1

            # 2. Check if repeat threshold exceeded
            if call_counts[call_hash] > max_repeats:
                print(
                    f"\n[LOOP DETECTED] Tool '{call.name}' called {call_counts[call_hash]} times with args: {call.arguments}"
                )
                print(">>> Breaking deliberately failing tool-retry cycle! <<<")
                return {
                    "status": "loop_detected",
                    "tool": call.name,
                    "arguments": call.arguments,
                    "repeats": call_counts[call_hash],
                }

            print(
                f"Tool requested: {call.name} (attempt {call_counts[call_hash]}) | Args: {call.arguments}"
            )
            try:
                result = TOOL_REGISTRY[call.name](**(call.arguments or {}))
                result_payload = {"ok": True, "result": result}
            except Exception as exc:  # noqa
                result_payload = {"ok": False, "error": str(exc)}

            print(f"Tool result: {result_payload}")

            tool_results.append(
                {
                    "type": "function_result",
                    "name": call.name,
                    "call_id": call.id,
                    "result": [{"type": "text", "text": json.dumps(result_payload)}],
                }
            )

        interaction = client.interactions.create(
            model=model,
            previous_interaction_id=interaction.id,
            input=tool_results,
            tools=TOOL_DEFINITIONS,
        )

    return {"status": "max_steps_exceeded"}

In [8]:
# Demonstrate loop detection breaking a deliberately failing tool-retry cycle
result = await run_agent_with_loop_detection(
    goal="Get the current weather in Paris using flaky_weather. If it fails, keep retrying.",
    client=google_client,
    model=MODEL,
    max_repeats=2,
)

print("\nFINAL AGENT RESULT:")
print(result)

Tool requested: flaky_weather (attempt 1) | Args: {'city': 'Paris'}
Tool result: {'ok': False, 'error': "Service 503: Weather service failed for 'Paris'. Please retry."}
Tool requested: flaky_weather (attempt 2) | Args: {'city': 'Paris'}
Tool result: {'ok': False, 'error': "Service 503: Weather service failed for 'Paris'. Please retry."}

[LOOP DETECTED] Tool 'flaky_weather' called 3 times with args: {'city': 'Paris'}
>>> Breaking deliberately failing tool-retry cycle! <<<

FINAL AGENT RESULT:
{'status': 'loop_detected', 'tool': 'flaky_weather', 'arguments': {'city': 'Paris'}, 'repeats': 3}


### Task 4

In [9]:
N_RUNS = 10
GOAL = "Get the CPU utilisation metric using get_metric."
VALID_KEY = "sys.cpu.util"

TOOL_DEFINITIONS_METRIC = [
    {
        "type": "function",
        "name": "get_metric",
        "description": "Fetch a system performance metric by internal key.",
        "parameters": {
            "type": "object",
            "properties": {
                "metric_key": {
                    "type": "string",
                    "description": "Internal metric key to retrieve.",
                }
            },
            "required": ["metric_key"],
        },
    }
]

# ── Three error-message variants ──────────────────────────────────────────────


def err_vague(key: str) -> str:
    return "Error: Unknown metric."


def err_specific(key: str) -> str:
    valid = ["sys.cpu.util", "sys.mem.rss", "sys.net.rx_bytes", "sys.disk.io_wait"]
    return f'Error: Unknown metric_key "{key}". Valid keys: {valid}.'


def err_specific_with_hint(key: str) -> str:
    valid = ["sys.cpu.util", "sys.mem.rss", "sys.net.rx_bytes", "sys.disk.io_wait"]
    return (
        f'Error: Unknown metric_key "{key}". Valid keys: {valid}. '
        f'Hint: For CPU usage use "sys.cpu.util".'
    )


ERROR_MODES: dict[str, Any] = {
    "vague": err_vague,
    "specific": err_specific,
    "specific_with_hint": err_specific_with_hint,
}

In [10]:
async def run_eval(error_fn: Any) -> bool:
    """
    Run one agent turn.
    Returns True if the agent corrects metric_key to VALID_KEY within 3 steps.
    """
    interaction = await asyncio.to_thread(
        google_client.interactions.create,
        model=MODEL,
        input=GOAL,
        tools=TOOL_DEFINITIONS_METRIC,
    )
    for _step in range(3):
        calls = [s for s in (interaction.steps or []) if s.type == "function_call"]
        if not calls:
            return False  # gave up without calling

        results = []
        for call in calls:
            key = str((call.arguments or {}).get("metric_key", ""))
            if key == VALID_KEY:
                return True  # recovered!
            payload = {"ok": False, "error": error_fn(key)}
            results.append(
                {
                    "type": "function_result",
                    "name": call.name,
                    "call_id": call.id,
                    "result": [{"type": "text", "text": json.dumps(payload)}],
                }
            )

        interaction = await asyncio.to_thread(
            google_client.interactions.create,
            model=MODEL,
            previous_interaction_id=interaction.id,
            input=results,
            tools=TOOL_DEFINITIONS_METRIC,
        )
    return False


async def benchmark(n_runs: int = N_RUNS) -> dict[str, int]:
    sem = asyncio.Semaphore(4)

    async def bounded(mode: str, error_fn: Any) -> bool:
        async with sem:
            return await run_eval(error_fn)

    results: dict[str, int] = {}
    for mode, error_fn in ERROR_MODES.items():
        print(f"Running {n_runs} evals for '{mode}' ...", flush=True)
        outcomes = await asyncio.gather(*[bounded(mode, error_fn) for _ in range(n_runs)])
        results[mode] = sum(outcomes)
        print(f"  → {results[mode]}/{n_runs} recovered")

    return results

In [11]:
scores = await benchmark()

# ── Pretty results table ──────────────────────────────────────────────────────
print()
print(f"{'Mode':<22} {'Recovered':>9}  {'Rate':>7}")
print("-" * 42)
for mode, n in scores.items():
    bar = "█" * n + "░" * (N_RUNS - n)
    rate = n / N_RUNS * 100
    print(f"{mode:<22} {n:>4}/{N_RUNS}    {rate:>5.0f}%  {bar}")

Running 10 evals for 'vague' ...
  → 0/10 recovered
Running 10 evals for 'specific' ...
  → 10/10 recovered
Running 10 evals for 'specific_with_hint' ...
  → 10/10 recovered

Mode                   Recovered     Rate
------------------------------------------
vague                     0/10        0%  ░░░░░░░░░░
specific                 10/10      100%  ██████████
specific_with_hint       10/10      100%  ██████████


### Task 5

implemented in test_approval_flow.py

### Task 6

### Task 7
implemented in test_prompt_injection.py

### Task 8

implemented in agent_langgrapg.py